# Draft room data by room ID — research spike

Goal: given only a Yahoo draft room ID, fetch the picks made so far — for both real leagues and public Live Standard mock drafts. NBA mock drafts are the test surface.

In [1]:
import json
import time

import requests
from yahoo_oauth import OAuth2
from yahoo_fantasy_api import Game, League

In [2]:
sc = OAuth2(None, None, from_file="../private.json")
print("token valid:", sc.token_is_valid())

[2026-09-21 20:18:39,146 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-09-21 20:18:39,149 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 949692.3887879848
[2026-09-21 20:18:39,149 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN HAS EXPIRED
[2026-09-21 20:18:39,150 DEBUG] [yahoo_oauth.oauth.refresh_access_token] REFRESHING TOKEN
[2026-09-21 20:18:39,661 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 0.5114860534667969
[2026-09-21 20:18:39,662 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


token valid: True


## Section A — official API attempt

Try the documented `league/{league_key}/draftresults` endpoint using candidate league_key formats built from a room ID.

In [ ]:
gm = Game(sc, 'nba')
game_key = gm.game_id()
print("game_key:", game_key)

In [ ]:
# Room ID comes from the draft client URL:
# https://basketball.fantasysports.yahoo.com/draftclient/nba/2479477/2 -> room_id = "2479477" (the trailing /2 is your team)
room_id = "2479477"

candidate_league_keys = [
    f"{game_key}.l.{room_id}",
    f"nba.l.{room_id}",
    room_id,
]

for candidate in candidate_league_keys:
    print(f"--- trying league_key={candidate!r} ---")
    try:
        lg = League(sc, candidate)
        raw = lg.yhandler.get_draftresults_raw(candidate)
        print("SUCCESS, raw response:")
        print(raw)
    except Exception as e:
        print(f"FAILED: {type(e).__name__}: {e}")

### Diagnostic — can this token reach real league data at all?

Before concluding the app registration type is the problem, test against an NBA league you actually belong to (known to work with this key). This isolates whether the token/app is broadly unauthorized, or whether the issue is specific to draft room access.

In [ ]:
gm_nba = Game(sc, 'nba')
print("nba game_key:", gm_nba.game_id())

nba_league_ids = gm_nba.league_ids()
print("your NBA league ids:", nba_league_ids)

### Diagnostic — discriminating ladder

Two different apps have now failed identically through `yahoo_fantasy_api`/`yahoo_oauth`. Before blaming
app config again, bypass the library entirely and call the API directly with `requests`, using only the
raw access token from `sc`. Run these four rungs in order against `https://fantasysports.yahooapis.com/fantasy/v2`:

1. `/game/nba` — public game metadata, lowest entitlement bar. If this 401s/403s, the problem is the
   credential or its scope, not any league key.
2. `/users;use_login=1/games` — proves the token resolves to a Yahoo user *and* that Fantasy scope is live.
   Failure here with step 1 passing means scope, not credentials.
3. `/users;use_login=1/games;game_keys=nba/leagues` — returns the league keys the authenticated account
   actually belongs to, with the current season's `game_key`. Authoritative source — never hardcode it.
4. `/league/{key}` using the key from step 3.

In [16]:
FANTASY_BASE = "https://fantasysports.yahooapis.com/fantasy/v2"


def yahoo_get(path):
    resp = requests.get(
        f"{FANTASY_BASE}/{path}",
        headers={
            "Authorization": f"Bearer {sc.access_token}",
            "Accept": "application/json",
        },
        params={"format": "json"},
    )
    print(f"--- GET {path} ---")
    print("status:", resp.status_code)
    print(resp.text[:2000])
    return resp


_ = yahoo_get("game/nba")

--- GET game/nba ---
status: 403
{
    "error": {
        "xml:lang": "en-us",
        "yahoo:uri": "\/fantasy\/v2\/game\/nba?format=json",
        "description": "This application is not authorized to perform this action.",
        "detail": ""
    }
}


In [17]:
_ = yahoo_get("users;use_login=1/games")

--- GET users;use_login=1/games ---
status: 403
{
    "error": {
        "xml:lang": "en-us",
        "yahoo:uri": "\/fantasy\/v2\/users;use_login=1\/games?format=json",
        "description": "This application is not authorized to perform this action.",
        "detail": ""
    }
}


In [18]:
leagues_resp = yahoo_get("users;use_login=1/games;game_keys=nba/leagues")

--- GET users;use_login=1/games;game_keys=nba/leagues ---
status: 403
{
    "error": {
        "xml:lang": "en-us",
        "yahoo:uri": "\/fantasy\/v2\/users;use_login=1\/games;game_keys=nba\/leagues?format=json",
        "description": "This application is not authorized to perform this action.",
        "detail": ""
    }
}


In [19]:
# Copy a league key (e.g. "nba.l.12345") out of the printed JSON from the previous cell.
league_key_from_step3 = "REPLACE_ME"

_ = yahoo_get(f"league/{league_key_from_step3}")

--- GET league/REPLACE_ME ---
status: 403
{
    "error": {
        "xml:lang": "en-us",
        "yahoo:uri": "\/fantasy\/v2\/league\/REPLACE_ME?format=json",
        "description": "This application is not authorized to perform this action.",
        "detail": ""
    }
}


### Finding — Section A works for mock drafts

`GET league/nba.l.{room_id}/draftresults` returns **200** for a Live Standard mock draft room (room `2479477`
resolved to `478.l.2479477`, `is_mock: 1`, `draft_status: draft`). The room ID *is* the league ID, so no
captured-request fallback (Section B) is needed. Adding the `/players` sub-resource inlines player names on
each pick; picks not yet made come back as empty lists.

## Section C — normalize + live-poll validation

Wrap the working Section A call in one function returning `[{round, pick, team, player, ...}]`, then confirm it
picks up new data while the draft is running.

In [ ]:
FANTASY_BASE = "https://fantasysports.yahooapis.com/fantasy/v2"
league_key = f"nba.l.{room_id}"


def fantasy_json(path):
    sc.token_is_valid()  # refreshes the access token if it has expired
    resp = requests.get(
        f"{FANTASY_BASE}/{path}",
        headers={"Authorization": f"Bearer {sc.access_token}"},
        params={"format": "json"},
    )
    if not resp.ok:
        print(f"GET {path} -> {resp.status_code}\n{resp.text[:1000]}")
    resp.raise_for_status()
    return resp.json()["fantasy_content"]


def merge_fields(fields):
    """Yahoo returns objects as a list of single-key dicts (with [] fillers); flatten into one dict."""
    merged = {}
    for f in fields:
        if isinstance(f, dict):
            merged.update(f)
    return merged


def get_team_names(league_key):
    teams = fantasy_json(f"league/{league_key}/teams")["league"][1]["teams"]
    names = {}
    for k, v in teams.items():
        if k == "count":
            continue
        t = merge_fields(v["team"][0])
        names[t["team_key"]] = t["name"]
    return names


def get_current_picks(league_key, team_names):
    """Fetch and normalize picks made so far into [{round, pick, team, player, pos, nba_team}, ...]."""
    results = fantasy_json(f"league/{league_key}/draftresults/players")["league"][1]["draft_results"]
    picks = []
    for k, v in results.items():
        if k == "count" or not v:  # unmade picks come back as empty lists
            continue
        dr = v["draft_result"]
        player = merge_fields(dr["0"]["players"]["0"]["player"][0])
        picks.append({
            "round": dr["round"],
            "pick": dr["pick"],
            "team": team_names.get(dr["team_key"], dr["team_key"]),
            "player": player["name"]["full"],
            "pos": player.get("display_position"),
            "nba_team": player.get("editorial_team_abbr"),
        })
    return sorted(picks, key=lambda p: p["pick"])

In [ ]:
league = fantasy_json(f"league/{league_key}")["league"][0]
print(league["name"], "| draft_status:", league["draft_status"], "| is_mock:", league["is_mock"], "| teams:", league["num_teams"])

team_names = get_team_names(league_key)
picks = get_current_picks(league_key, team_names)
print(f"{len(picks)} picks so far")
for p in picks:
    print(f"R{p['round']:>2} #{p['pick']:>3}  {p['team']:<15} {p['player']} ({p['pos']}, {p['nba_team']})")

In [ ]:
# Run while the draft is actively picking: prints only picks that are new since the last check.
seen = {p["pick"] for p in picks}
for _ in range(8):  # ~2 min at 15s intervals
    time.sleep(15)
    new = [p for p in get_current_picks(league_key, team_names) if p["pick"] not in seen]
    if new:
        for p in new:
            print(f"NEW  R{p['round']:>2} #{p['pick']:>3}  {p['team']:<15} {p['player']} ({p['pos']}, {p['nba_team']})")
        seen.update(p["pick"] for p in new)
    else:
        print(f"no new picks ({len(seen)} total)")